Convert the SOFT to csv for easy visualization
This notebook has the propuse of familiriaze me with the data we have. 
This will clean and place the series matrix and SOFT file to see them on CSV because is the way I like to see them.

In [1]:
import re
import pandas as pd

def parse_soft_file(soft_file_path):
    """Parses a GEO SOFT file and extracts sample metadata."""

    sample_data = []
    current_sample = {}
    reading_sample = False

    with open(soft_file_path, 'r') as infile:
        for line in infile:
            line = line.strip()

            # Detect start of a new sample record
            if line.startswith("^SAMPLE = GSM"):
                if current_sample:
                    sample_data.append(current_sample)
                current_sample = {'GSM': line.split("= ")[1]}  # Initialize with GSM ID
                reading_sample = True
                continue # Skip the rest of the loop and proceed to the next line

            if reading_sample:
                # Extract metadata
                if   "= " in line:
                    parts = line.split("= ")
                    key = parts[0].lstrip("!Sample_")
                    value = parts[1]
                    current_sample[key] = value
                # Handle characteristics annotations

    # Append the last sample
    if current_sample:
        sample_data.append(current_sample)

    return sample_data

def soft_to_dataframe(soft_file_path):
    """Parses SOFT file and returns a pandas DataFrame."""
    data = parse_soft_file(soft_file_path)
    df = pd.DataFrame(data)
    return df



In [11]:
soft_file = "Data/GSE96058_family.soft" 
df = soft_to_dataframe(soft_file)
df = df.loc[:, (df != df.iloc[0]).any()]
df.to_csv("Data/GSE96058_metadata.csv", index=False)  # Save as CSV
print(df.head())

          GSM title  geo_accession  st_update_date  characteristics_ch1   \
0  GSM2528079     F1     GSM2528079     May 04 2022     chemo treated: 1   
1  GSM2528080     F2     GSM2528080     May 04 2022     chemo treated: 1   
2  GSM2528081     F3     GSM2528081     May 04 2022     chemo treated: 1   
3  GSM2528082     F4     GSM2528082     May 04 2022     chemo treated: 1   
4  GSM2528083     F5     GSM2528083     May 04 2022     chemo treated: 0   

  tform_id     instrument_model   \
0  GPL11154  Illumina HiSeq 2000   
1  GPL11154  Illumina HiSeq 2000   
2  GPL11154  Illumina HiSeq 2000   
3  GPL11154  Illumina HiSeq 2000   
4  GPL11154  Illumina HiSeq 2000   

                                           relation   
0  BioSample: https://www.ncbi.nlm.nih.gov/biosam...  
1  BioSample: https://www.ncbi.nlm.nih.gov/biosam...  
2  BioSample: https://www.ncbi.nlm.nih.gov/biosam...  
3  BioSample: https://www.ncbi.nlm.nih.gov/biosam...  
4  BioSample: https://www.ncbi.nlm.nih.gov/biosam..

In [22]:
import pandas as pd

def read_series_matrix(file_path):
    """Reads a GEO series matrix file into a pandas DataFrame."""
    with open(file_path, 'r') as f:
        # Determine number of comment lines to skip (starting with "!")
        comment_lines = 0
        for line in f:
            if line.startswith("!"):
                comment_lines += 1
            else:
                break

    # Read data, skipping comment lines
    df = pd.read_csv(file_path, sep='\t', skiprows=comment_lines, index_col=0)
    return df



In [63]:
def clean_series_matrix(df):
    """
    Cleans a GEO series matrix DataFrame by extracting column names
    and values from "name: value" formatted columns.
    """
    column_names=[]
    for col in df.columns:
        # Check if "Sample_characteristics_ch1" is in the column name
        if "!Sample_characteristics_ch1" in col:
            # Extract example value, we will extract the name from the fist row
            example_value = str(df[col].iloc[0])
            print(f"example value: {example_value}")
            name = example_value.split(": ")[0]
            column_names.append(name)
        else:
            column_names.append(col)
    #print(column_names)
    return df
        


In [73]:

first_row = expression_df.iloc[0]
column_names=[]
for i, value in enumerate(first_row):
         if  isinstance(value, str) and ": " in value:
            column_names.append(value.split(": ")[0])
         else:
            #If the name does not exist, keept the same as before
            column_names.append(expression_df.columns[i].replace("!",""))
column_names

['Sample_geo_accession',
 'Sample_last_update_date',
 'scan-b external id',
 'age at diagnosis',
 'tumor size',
 'lymph node group',
 'lymph node status',
 'er status',
 'pgr status',
 'her2 status',
 'ki67 status',
 'nhg',
 'er prediction mgc',
 'pgr prediction mgc',
 'her2 prediction mgc',
 'ki67 prediction mgc',
 'nhg prediction mgc',
 'er prediction sgc',
 'pgr prediction sgc',
 'her2 prediction sgc',
 'ki67 prediction sgc',
 'pam50 subtype',
 'overall survival days',
 'overall survival event',
 'endocrine treated',
 'chemo treated',
 'Reanalyzed by',
 'BioSample',
 'series_matrix_table_begin',
 'ID_REF',
 'series_matrix_table_end']

In [ ]:
name = "GSE96058-GPL11154_series_matrix.txt"
matrix_file = f"Data/{name}" 
expression_df = read_series_matrix(matrix_file)
expression_df = expression_df.T
expression_df = expression_df.loc[:, (expression_df != expression_df.iloc[0]).any()]

In [75]:
expression_df.columns = column_names

In [80]:
for col in expression_df.columns:
    expression_df[col] = expression_df[col].astype(str).apply(lambda x: x.split(": ", 1)[1] if ": " in x else x)

expression_df


,Sample_geo_accession,Sample_last_update_date,scan-b external id,age at diagnosis,tumor size,lymph node group,lymph node status,er status,pgr status,her2 status,...,pam50 subtype,overall survival days,overall survival event,endocrine treated,chemo treated,Reanalyzed by,BioSample,series_matrix_table_begin,ID_REF,series_matrix_table_end
F1,GSM2528079,May 04 2022,Q008818.C008840.S000215.l.r.m2.c.lib.g.k.a.t,43,9,NodeNegative,NodeNegative,NA,NA,0,...,Basal,2367,0,0,1,GSM6103185,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,nan,GSM2528079,nan
F2,GSM2528080,May 04 2022,Q008769.C008792.S000250.l.r.m.c.lib.g.k.a.t,48,14,1to3,NodePositive,1,1,0,...,LumA,2367,0,1,1,GSM6103213,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,nan,GSM2528080,nan
F3,GSM2528081,May 04 2022,Q008568.C008577.S000424.l.r.m3.c.lib.g.k.a.t,69,27,4toX,NodePositive,1,1,0,...,LumB,2168,1,1,1,GSM6103344,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,nan,GSM2528081,nan
F4,GSM2528082,May 04 2022,Q008909.C009000.S000084.l.r.m.c.lib.g.k.a.t,39,51,1to3,NodePositive,1,NA,1,...,LumA,2416,0,1,1,GSM6103091,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,nan,GSM2528082,nan
F5,GSM2528083,May 04 2022,Q008781.C008782.S000260.l.r.m.c.lib.g.k.a.t,73,60,4toX,NodePositive,1,NA,0,...,Normal,2389,0,1,0,GSM6103222,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,nan,GSM2528083,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
F2912repl,GSM2531475,Mar 12 2018,Q006763.C006741.S002377.l.r.m.c.lib.g.k.a.t,75,19,1to3,NodePositive,1,NA,0,...,LumA,490,1,1,0,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,nan,nan,GSM2531475,nan
F2958repl,GSM2531477,Mar 12 2018,Q005521.C005590.S003572.l.r.m2.c.lib.g.k.a.t,33,12,NA,NA,NA,NA,1,...,Her2,1502,0,1,1,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,nan,nan,GSM2531477,nan
F3057repl,GSM2531481,Mar 12 2018,Q005327.C005381.S003761.l2.r.m.c.lib.g.k.a.t,76,91,NodeNegative,NodeNegative,1,0,0,...,LumA,1473,0,0,0,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,nan,nan,GSM2531481,nan
F3085repl,GSM2531483,Mar 12 2018,Q005150.C005183.S003987.l.r.m.c.lib.g.k.a.t,79,10,1to3,NodePositive,1,1,0,...,LumB,1426,0,1,0,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,nan,nan,GSM2531483,nan


In [81]:
expression_df = expression_df.loc[:, (expression_df != expression_df.iloc[0]).any()]
expression_df.head()

,Sample_geo_accession,Sample_last_update_date,scan-b external id,age at diagnosis,tumor size,lymph node group,lymph node status,er status,pgr status,her2 status,...,her2 prediction sgc,ki67 prediction sgc,pam50 subtype,overall survival days,overall survival event,endocrine treated,chemo treated,Reanalyzed by,BioSample,ID_REF
F1,GSM2528079,May 04 2022,Q008818.C008840.S000215.l.r.m2.c.lib.g.k.a.t,43,9,NodeNegative,NodeNegative,NA,NA,0,...,0,1,Basal,2367,0,0,1,GSM6103185,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,GSM2528079
F2,GSM2528080,May 04 2022,Q008769.C008792.S000250.l.r.m.c.lib.g.k.a.t,48,14,1to3,NodePositive,1,1,0,...,0,0,LumA,2367,0,1,1,GSM6103213,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,GSM2528080
F3,GSM2528081,May 04 2022,Q008568.C008577.S000424.l.r.m3.c.lib.g.k.a.t,69,27,4toX,NodePositive,1,1,0,...,0,1,LumB,2168,1,1,1,GSM6103344,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,GSM2528081
F4,GSM2528082,May 04 2022,Q008909.C009000.S000084.l.r.m.c.lib.g.k.a.t,39,51,1to3,NodePositive,1,NA,1,...,1,1,LumA,2416,0,1,1,GSM6103091,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,GSM2528082
F5,GSM2528083,May 04 2022,Q008781.C008782.S000260.l.r.m.c.lib.g.k.a.t,73,60,4toX,NodePositive,1,NA,0,...,0,0,Normal,2389,0,1,0,GSM6103222,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,GSM2528083


In [82]:

expression_df.to_csv(f"Data/{name}.csv", index=False)  # Save as CSV

expression_df

,Sample_geo_accession,Sample_last_update_date,scan-b external id,age at diagnosis,tumor size,lymph node group,lymph node status,er status,pgr status,her2 status,...,her2 prediction sgc,ki67 prediction sgc,pam50 subtype,overall survival days,overall survival event,endocrine treated,chemo treated,Reanalyzed by,BioSample,ID_REF
F1,GSM2528079,May 04 2022,Q008818.C008840.S000215.l.r.m2.c.lib.g.k.a.t,43,9,NodeNegative,NodeNegative,NA,NA,0,...,0,1,Basal,2367,0,0,1,GSM6103185,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,GSM2528079
F2,GSM2528080,May 04 2022,Q008769.C008792.S000250.l.r.m.c.lib.g.k.a.t,48,14,1to3,NodePositive,1,1,0,...,0,0,LumA,2367,0,1,1,GSM6103213,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,GSM2528080
F3,GSM2528081,May 04 2022,Q008568.C008577.S000424.l.r.m3.c.lib.g.k.a.t,69,27,4toX,NodePositive,1,1,0,...,0,1,LumB,2168,1,1,1,GSM6103344,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,GSM2528081
F4,GSM2528082,May 04 2022,Q008909.C009000.S000084.l.r.m.c.lib.g.k.a.t,39,51,1to3,NodePositive,1,NA,1,...,1,1,LumA,2416,0,1,1,GSM6103091,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,GSM2528082
F5,GSM2528083,May 04 2022,Q008781.C008782.S000260.l.r.m.c.lib.g.k.a.t,73,60,4toX,NodePositive,1,NA,0,...,0,0,Normal,2389,0,1,0,GSM6103222,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,GSM2528083
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
F2912repl,GSM2531475,Mar 12 2018,Q006763.C006741.S002377.l.r.m.c.lib.g.k.a.t,75,19,1to3,NodePositive,1,NA,0,...,0,1,LumA,490,1,1,0,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,nan,GSM2531475
F2958repl,GSM2531477,Mar 12 2018,Q005521.C005590.S003572.l.r.m2.c.lib.g.k.a.t,33,12,NA,NA,NA,NA,1,...,1,1,Her2,1502,0,1,1,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,nan,GSM2531477
F3057repl,GSM2531481,Mar 12 2018,Q005327.C005381.S003761.l2.r.m.c.lib.g.k.a.t,76,91,NodeNegative,NodeNegative,1,0,0,...,0,0,LumA,1473,0,0,0,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,nan,GSM2531481
F3085repl,GSM2531483,Mar 12 2018,Q005150.C005183.S003987.l.r.m.c.lib.g.k.a.t,79,10,1to3,NodePositive,1,1,0,...,0,1,LumB,1426,0,1,0,https://www.ncbi.nlm.nih.gov/biosample/SAMN065...,nan,GSM2531483


In [76]:
expression_df

,Sample_geo_accession,Sample_last_update_date,scan-b external id,age at diagnosis,tumor size,lymph node group,lymph node status,er status,pgr status,her2 status,...,pam50 subtype,overall survival days,overall survival event,endocrine treated,chemo treated,Reanalyzed by,BioSample,series_matrix_table_begin,ID_REF,series_matrix_table_end
F1,GSM2528079,May 04 2022,scan-b external id: Q008818.C008840.S000215.l....,age at diagnosis: 43,tumor size: 9,lymph node group: NodeNegative,lymph node status: NodeNegative,er status: NA,pgr status: NA,her2 status: 0,...,pam50 subtype: Basal,overall survival days: 2367,overall survival event: 0,endocrine treated: 0,chemo treated: 1,Reanalyzed by: GSM6103185,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,NaN,GSM2528079,NaN
F2,GSM2528080,May 04 2022,scan-b external id: Q008769.C008792.S000250.l....,age at diagnosis: 48,tumor size: 14,lymph node group: 1to3,lymph node status: NodePositive,er status: 1,pgr status: 1,her2 status: 0,...,pam50 subtype: LumA,overall survival days: 2367,overall survival event: 0,endocrine treated: 1,chemo treated: 1,Reanalyzed by: GSM6103213,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,NaN,GSM2528080,NaN
F3,GSM2528081,May 04 2022,scan-b external id: Q008568.C008577.S000424.l....,age at diagnosis: 69,tumor size: 27,lymph node group: 4toX,lymph node status: NodePositive,er status: 1,pgr status: 1,her2 status: 0,...,pam50 subtype: LumB,overall survival days: 2168,overall survival event: 1,endocrine treated: 1,chemo treated: 1,Reanalyzed by: GSM6103344,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,NaN,GSM2528081,NaN
F4,GSM2528082,May 04 2022,scan-b external id: Q008909.C009000.S000084.l....,age at diagnosis: 39,tumor size: 51,lymph node group: 1to3,lymph node status: NodePositive,er status: 1,pgr status: NA,her2 status: 1,...,pam50 subtype: LumA,overall survival days: 2416,overall survival event: 0,endocrine treated: 1,chemo treated: 1,Reanalyzed by: GSM6103091,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,NaN,GSM2528082,NaN
F5,GSM2528083,May 04 2022,scan-b external id: Q008781.C008782.S000260.l....,age at diagnosis: 73,tumor size: 60,lymph node group: 4toX,lymph node status: NodePositive,er status: 1,pgr status: NA,her2 status: 0,...,pam50 subtype: Normal,overall survival days: 2389,overall survival event: 0,endocrine treated: 1,chemo treated: 0,Reanalyzed by: GSM6103222,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,NaN,GSM2528083,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
F2912repl,GSM2531475,Mar 12 2018,scan-b external id: Q006763.C006741.S002377.l....,age at diagnosis: 75,tumor size: 19,lymph node group: 1to3,lymph node status: NodePositive,er status: 1,pgr status: NA,her2 status: 0,...,pam50 subtype: LumA,overall survival days: 490,overall survival event: 1,endocrine treated: 1,chemo treated: 0,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,NaN,NaN,GSM2531475,NaN
F2958repl,GSM2531477,Mar 12 2018,scan-b external id: Q005521.C005590.S003572.l....,age at diagnosis: 33,tumor size: 12,lymph node group: NA,lymph node status: NA,er status: NA,pgr status: NA,her2 status: 1,...,pam50 subtype: Her2,overall survival days: 1502,overall survival event: 0,endocrine treated: 1,chemo treated: 1,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,NaN,NaN,GSM2531477,NaN
F3057repl,GSM2531481,Mar 12 2018,scan-b external id: Q005327.C005381.S003761.l2...,age at diagnosis: 76,tumor size: 91,lymph node group: NodeNegative,lymph node status: NodeNegative,er status: 1,pgr status: 0,her2 status: 0,...,pam50 subtype: LumA,overall survival days: 1473,overall survival event: 0,endocrine treated: 0,chemo treated: 0,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,NaN,NaN,GSM2531481,NaN
F3085repl,GSM2531483,Mar 12 2018,scan-b external id: Q005150.C005183.S003987.l....,age at diagnosis: 79,tumor size: 10,lymph node group: 1to3,lymph node status: NodePositive,er status: 1,pgr status: 1,her2 status: 0,...,pam50 subtype: LumB,overall 

In [5]:
matrix_file = "GSE96058-GPL11154_series_matrix.txt"  # Replace with your file path


""
